# 09. 08 의 남은 오류를 겨냥한 개선 (고정 풀기 + 후보 넓히기 + 언어모델 미세조정)
08 (검증 0.9894) 에 남은 글자 오류 1,172개를 분해한 결과:

| 원인 | 비중 | 정답 어절이 train 에 없는 비율 |
|---|---|---|
| ① 고정했는데 틀림 (01·v2 가 같이 틀림) | 42.5% | 60% |
| ② 후보에 정답이 없음 | 38.7% | 91% |
| ③ 후보에 정답이 있는데 언어모델이 잘못 고름 | 18.8% | 60% |

남은 오류 대부분이 train 에 없는 단어라서, train 빈도 가점·01 유지 문턱(1차 09 시도)은 효과가 없었다.

| 개선 | 겨냥 | 방법 | 로컬 확인 |
|---|---|---|---|
| **A** 의심스러운 고정 풀기 | ① | 01 의 답이 train 에 없는 어절이면 일치해도 추론 대상에 (`--unanchor_oov`) | 그런 고정 어절 정확도 90.3% (10회 이상 나온 어절은 99.1%) |
| **B** 후보 넓히기 | ② | 후보 생성 빔 64, 후보 20개 (`--cand_beam 64 --nbest 20`) | 후보 상한 0.9929 → 0.9950 |
| **C** 언어모델 미세조정 | ③ + 늘어난 선택 | Qwen2.5-1.5B 를 train 정답 리뷰로 LoRA 미세조정 (`lm_finetune.py`) | 미실행 |

각 실행은 `--name` 으로 따로 저장되어 서로 덮어쓰지 않는다. 기록은 `outputs/anchor_runs.csv`.

**GPU 런타임(L4 권장), 첫 셀부터.**

## 0) 준비

In [ ]:
import os
os.chdir('/content')
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DRIVE = '/content/drive/MyDrive/데이콘/난독화'
!mkdir -p /content/work && rsync -a --exclude '.playwright-mcp' --exclude '__pycache__' --exclude 'open.zip' --exclude 'v2_cache_*.pkl' "$DRIVE/" /content/work/
%cd /content/work
!ls outputs | grep -E 'val_pred.csv|val_pred_tagger_v2_scratch|submission_tagger_v2_scratch|anchor_items_val_agree_v2_k6.pkl|lm_adapter'; nproc; nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
def save_back():
    if not os.path.exists(DRIVE):
        drive.mount('/content/drive', force_remount=True)
    !rsync -a --update --exclude 'v2_cache_*.pkl' /content/work/outputs/ "$DRIVE/outputs/"
    print('saved outputs -> drive')

## 1) C: 공개 언어모델 미세조정 (검증용: train 90% 의 정답 리뷰만)
출력의 '검증 리뷰 손실' 이 학습 전보다 내려가면 리뷰 말투를 배운 것.
병합 모델은 `models/qwen_ft_split` (약 3GB, 로컬만). 어댑터는 `outputs/lm_adapter_split` 에 백업.
런타임이 바뀌어 `models/` 가 사라졌으면: `!python -u lm_finetune.py --merge_only`

In [ ]:
!pip -q install peft
!pip -q uninstall -y torchao
!python -u lm_finetune.py --merge_only


## 2) 효과를 하나씩 확인 (검증 전체)
첫 w 는 언어모델 점수를 처음부터 계산하고, 이후 w 는 캐시로 빨라진다. 08 기본은 0.9894 (w 1.5).

### 2-1) C 만: 08 기본에 미세조정 모델만 교체 (약 5분)

In [ ]:
!python -u anchor_infill.py --mode val --weights 1,1.5,2 --lm models/qwen_ft_split --name 09c
save_back()

### 2-2) A 만: 의심스러운 고정 풀기 (약 5~8분)

In [ ]:
!python -u anchor_infill.py --mode val --weights 1,1.5,2 --unanchor_oov --name 09a
save_back()

### 2-3) A + B: 후보 넓히기까지 (약 15~25분)
메모리 부족 에러가 나면 `--batch_tokens 8192`.

In [ ]:
!python -u anchor_infill.py --mode val --weights 1,1.5,2 --unanchor_oov --cand_beam 64 --nbest 20 --batch_tokens 8192 --name 09ab
save_back()

### 2-4) A + B + C: 전부 (약 15~25분)

In [ ]:
!python -u anchor_infill.py --mode val --weights 1,1.5,2 --unanchor_oov --cand_beam 64 --nbest 20 --batch_tokens 8192 --lm models/qwen_ft_split --name 09abc
save_back()

In [ ]:
!python -u anchor_infill.py --mode val --weights 0.5,0.6,0.7,0.8,0.9,1.0,1.1,1.25 --unanchor_oov --cand_beam 64 --nbest 20 --batch_tokens 32768 --lm models/qwen_ft_split --name 09abc
save_back()


## 3) 결과 표

In [ ]:
import pandas as pd
r = pd.read_csv('outputs/anchor_runs.csv')
r = r[r.n_val == r.n_val.max()].copy()
r['name'] = r['name'].fillna('08')
best = r.sort_values('f1', ascending=False).groupby('name').head(1)
print('실험별 최고:'); display(best[['name', 'lm', 'nbest', 'w', 'f1', 'oracle_f1', 'sec']])
r.sort_values('f1', ascending=False)[['name', 'lm', 'nbest', 'w', 'w_freq', 'margin', 'f1', 'oracle_f1', 'sec']].head(25)

## 4) 01 / 08 / 09 비교 + 예시

In [ ]:
import glob
a = pd.read_csv('outputs/val_pred.csv').set_index('ID')
base = pd.read_csv('outputs/val_pred_anchor_agree_v2_k6.csv').set_index('ID')
print(f"01            F1 {a.f1.mean():.4f}\n08            F1 {base.f1.mean():.4f}")
for f in sorted(glob.glob('outputs/val_pred_anchor_*_09*.csv')):
    b = pd.read_csv(f).set_index('ID'); b['pred'] = b['pred'].fillna('')
    d = b.f1 - base.f1
    name = f.split('_')[-1][:-4]
    print(f"{name:13s} F1 {b.f1.mean():.4f} | 08 대비 좋아진 행 {(d>0).sum()} / 나빠진 행 {(d<0).sum()}")
    if name == '09abc':
        for i in d.sort_values().index[:3].tolist() + d.sort_values().index[-4:].tolist():
            print(f'   [{d[i]:+.3f}] GT :', b.loc[i, 'output'][:90]); print('            08 :', base.loc[i, 'pred'][:90]); print('            09 :', b.loc[i, 'pred'][:90])

## 5) test 예측 (가장 좋은 조합이 A+B+C 일 때의 예)
test 용 언어모델은 train **전체**로 다시 미세조정한다. 옵션과 `--name` 은 검증에서 고른 조합과 같게.
다른 조합이 최고였다면 옵션을 그에 맞게 바꿀 것 (예: 09ab 는 `--lm` 줄 삭제).

In [ ]:
!python -u lm_finetune.py --full --batch_tokens 4096
!python -u anchor_infill.py --mode test --unanchor_oov --cand_beam 64 --nbest 20 --batch_tokens 32768 --lm models/qwen_ft_full --name 09abc
save_back()


### (선택) 런타임 해제

In [ ]:
from google.colab import runtime; runtime.unassign()